In [1]:
import sys, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from black_scholes import bs_call, bs_put
from greeks import delta as bs_delta, gamma as bs_gamma, vega as bs_vega, theta as bs_theta

os.makedirs('../figures', exist_ok=True)

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
spots = np.linspace(50, 150, 300)
sns.set_theme(style='darkgrid')
print('Setup OK — params: S={}, K={}, T={}, r={}, sigma={}'.format(S0, K, T, r, sigma))


Setup OK — params: S=100.0, K=100.0, T=1.0, r=0.05, sigma=0.2


## 1. Prix Call & Put vs Spot

Le prix Black-Scholes est toujours **au-dessus du payoff à maturité** (valeur temps > 0).
Quand S → ∞, le call converge vers le forward ; quand S → 0, le put tend vers K·e^{-rT}.
Les pointillés montrent la valeur intrinsèque à maturité (payoff).


In [2]:
calls = [bs_call(s, K, T, r, sigma) for s in spots]
puts  = [bs_put(s,  K, T, r, sigma) for s in spots]
payoff_call = np.maximum(spots - K, 0)
payoff_put  = np.maximum(K - spots, 0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(spots, calls, label='Call BS',               color='steelblue', lw=2)
ax.plot(spots, puts,  label='Put BS',                color='tomato',    lw=2)
ax.plot(spots, payoff_call, '--', label='Payoff call (maturité)', color='steelblue', alpha=0.5)
ax.plot(spots, payoff_put,  '--', label='Payoff put  (maturité)', color='tomato',    alpha=0.5)
ax.axvline(K, color='gray', linestyle=':', lw=1.2, label=f'ATM (K={K})')
ax.set_title('Prix Call et Put vs Spot — Black-Scholes', fontsize=13)
ax.set_xlabel('Spot S (€)')
ax.set_ylabel('Prix (€)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/01_price_vs_spot.png', dpi=150)
plt.show()
print('Saved: figures/01_price_vs_spot.png')


Saved: figures/01_price_vs_spot.png


C:\Users\octav\AppData\Local\Temp\ipykernel_26944\195068621.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Delta vs Spot

Le **delta** mesure l'exposition directionnelle : combien la valeur du portefeuille varie
pour +1 € sur le sous-jacent. ATM, le delta call ≈ 0.64 et non 0.5 à cause du drift r.
La couverture delta-neutre consiste à détenir −Δ unités du sous-jacent pour chaque option.


In [3]:
delta_calls = [bs_delta(s, K, T, r, sigma, 'call') for s in spots]
delta_puts  = [bs_delta(s, K, T, r, sigma, 'put')  for s in spots]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(spots, delta_calls, label='Delta Call', color='steelblue', lw=2)
ax.plot(spots, delta_puts,  label='Delta Put',  color='tomato',    lw=2)
ax.axvline(K, color='gray', linestyle=':', lw=1.5, label=f'ATM (S=K={K})')
ax.axhline(0, color='black', lw=0.7, linestyle='-')
ax.set_title('Delta vs Spot', fontsize=13)
ax.set_xlabel('Spot S (€)')
ax.set_ylabel('Delta')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/02_delta_vs_spot.png', dpi=150)
plt.show()
print('Saved: figures/02_delta_vs_spot.png')


Saved: figures/02_delta_vs_spot.png


C:\Users\octav\AppData\Local\Temp\ipykernel_26944\1838391766.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Gamma & Vega vs Spot

**Gamma** (d²V/dS²) : vitesse de variation du delta — maximal ATM, nul deep ITM/OTM.
**Vega** (dV/dσ) : sensibilité à la volatilité implicite — aussi maximal ATM.
Ces deux Greeks ont exactement la même forme de cloche centrée sur le strike.


In [4]:
gammas = [bs_gamma(s, K, T, r, sigma) for s in spots]
vegas  = [bs_vega(s,  K, T, r, sigma) for s in spots]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(spots, gammas, color='purple', lw=2)
ax1.axvline(K, color='gray', linestyle=':', lw=1.5, label=f'ATM (K={K})')
ax1.set_title('Gamma vs Spot', fontsize=13)
ax1.set_xlabel('Spot S (€)')
ax1.set_ylabel('Gamma')
ax1.legend()

ax2.plot(spots, vegas, color='seagreen', lw=2)
ax2.axvline(K, color='gray', linestyle=':', lw=1.5, label=f'ATM (K={K})')
ax2.set_title('Vega vs Spot (par +1 % de vol)', fontsize=13)
ax2.set_xlabel('Spot S (€)')
ax2.set_ylabel('Vega (€ / 1% vol)')
ax2.legend()

plt.tight_layout()
plt.savefig('../figures/03_gamma_vega_vs_spot.png', dpi=150)
plt.show()
print('Saved: figures/03_gamma_vega_vs_spot.png')


Saved: figures/03_gamma_vega_vs_spot.png


C:\Users\octav\AppData\Local\Temp\ipykernel_26944\4070292836.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Theta du Call vs Maturité

Le **theta** représente l'érosion temporelle quotidienne de la valeur temps.
Il est toujours négatif (acheteur d'option "perd" de la valeur chaque jour).
L'accélération vers T → 0 est caractéristique : les options courtes perdent
leur valeur très rapidement (*time decay* exponentiel en fin de vie).


In [5]:
maturities  = np.linspace(0.01, 2, 300)
thetas_call = [bs_theta(S0, K, t, r, sigma, 'call') for t in maturities]
thetas_put  = [bs_theta(S0, K, t, r, sigma, 'put')  for t in maturities]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(maturities, thetas_call, color='darkorange', lw=2, label='Theta Call')
ax.plot(maturities, thetas_put,  color='steelblue',  lw=2, label='Theta Put',  linestyle='--')
ax.axhline(0, color='black', lw=0.7)
ax.set_title('Theta vs Maturité — érosion temporelle (ATM, S=100)', fontsize=13)
ax.set_xlabel('Maturité T (années)')
ax.set_ylabel('Theta (€ / jour)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/04_theta_vs_maturity.png', dpi=150)
plt.show()
print('Saved: figures/04_theta_vs_maturity.png')


Saved: figures/04_theta_vs_maturity.png


C:\Users\octav\AppData\Local\Temp\ipykernel_26944\54105829.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Surface de prix 3D — Call Européen

Prix du call en fonction du **Spot S** et de la **Maturité T**.
La surface montre que la valeur temps croît avec T (surface convexe vers le haut)
et que l'option est plus sensible au spot pour les longues maturités.


In [6]:
spot_range = np.linspace(50, 150, 60)
mat_range  = np.linspace(0.05, 2, 60)
SS, TT = np.meshgrid(spot_range, mat_range)
ZZ = np.vectorize(lambda s, t: bs_call(s, K, t, r, sigma))(SS, TT)

fig3d = go.Figure(data=[go.Surface(
    x=spot_range,
    y=mat_range,
    z=ZZ,
    colorscale='Viridis',
    showscale=True,
    colorbar=dict(title='Prix Call (€)'),
)])
fig3d.update_layout(
    title='Surface de prix — Call Européen Black-Scholes',
    scene=dict(
        xaxis_title='Spot S (€)',
        yaxis_title='Maturité T (années)',
        zaxis_title='Prix Call (€)',
    ),
    width=850,
    height=620,
)
fig3d.write_html('../figures/05_call_surface_3d.html')
print('Saved: figures/05_call_surface_3d.html')
fig3d.show()


Saved: figures/05_call_surface_3d.html
